In [19]:
import pandas as pd

# Lire un fichier CSV et créer un DataFrame
df = pd.read_csv('activites_brutes.csv')

# Afficher les premières lignes
print(df.head())

# Afficher les informations du DataFrame
print(df.info())

   resource_state                                 athlete  \
0               2  {'id': 124945657, 'resource_state': 1}   
1               2  {'id': 124945657, 'resource_state': 1}   
2               2  {'id': 124945657, 'resource_state': 1}   
3               2  {'id': 124945657, 'resource_state': 1}   
4               2  {'id': 124945657, 'resource_state': 1}   

                      name  distance  moving_time  elapsed_time  \
0    Course à pied de nuit    7547.9         1822          1833   
1          10*400 moy 3:01    9561.0         2283          2496   
2          10*400 Moy 3:01   12299.0         2828          3084   
3    Course à pied de nuit   16425.0         3717          3881   
4  Course à pied en soirée    8721.5         2130          2425   

   total_elevation_gain type sport_type  workout_type  ... elev_high  \
0                  73.0  Run        Run           NaN  ...     102.0   
1                  13.0  Run        Run           NaN  ...     112.0   
2             

Analyse globale des éléments 


In [20]:
nb_colonnes = df.shape[1]
nb_lignes = df.shape[0]
print(f"Nombre de colonnes : {nb_colonnes}")
print(f"Nombre de lignes : {nb_lignes}") # on affiche le nombre de lignes celle-ci sont au nombre de 200. L'extraction entraine une 
# Une perte d'information. 
# Création d'un petit tableau récapitulatif
types_colonnes = pd.DataFrame({
    'Type': df.dtypes,
    'Valeurs non-nulles': df.count(),
    'Exemple': df.iloc[0] if len(df) > 0 else "N/A"
})

print(types_colonnes)

Nombre de colonnes : 57
Nombre de lignes : 571
                                  Type  Valeurs non-nulles  \
resource_state                   int64                 571   
athlete                            str                 571   
name                               str                 571   
distance                       float64                 571   
moving_time                      int64                 571   
elapsed_time                     int64                 571   
total_elevation_gain           float64                 571   
type                               str                 571   
sport_type                         str                 571   
workout_type                   float64                  98   
device_name                        str                 571   
id                               int64                 571   
start_date                         str                 571   
start_date_local                   str                 571   
timezone               

Suppression des colonnes unutile 

In [41]:
# 1. Ta sélection de colonnes (Zone Silver)
colonnes_souhaitees = [
    'start_date_local', 'name', 'type', 'distance', 'moving_time', 
    'total_elevation_gain', 'average_speed', 'max_speed', 
    'average_heartrate', 'max_heartrate', 'suffer_score', 
    'location_city', 'average_cadence', 'average_temp', 'kilojoules'
]

# 2. Filtrage intelligent des activités (Run uniquement)
# Liste des types officiels Strava pour la course
types_run_officiels = ['Run', 'TrailRun', 'VirtualRun']

# Liste de mots-clés pour identifier la course dans le nom (au cas où)
mots_cles_run = ['run', 'course', 'trail', '400', 'séance', 'entraînement']

# On filtre : soit le type est 'Run', soit le nom contient un mot-clé de course
# (en ignorant les majuscules avec .str.lower())
masque_run = (df['type'].isin(types_run_officiels)) | \
             (df['name'].str.lower().str.contains('|'.join(mots_cles_run), na=False))

# On exclut spécifiquement le vélo et la muscu même s'ils ont un mot-clé
masque_exclure = df['name'].str.lower().str.contains('vélo|musculation|poids|randonnée|marche', na=False)

df_propre = df[masque_run & ~masque_exclure].copy()

# 3. Application de ta sélection de colonnes
colonnes_finales = [c for c in colonnes_souhaitees if c in df_propre.columns]
df_propre = df_propre[colonnes_finales]



In [42]:
print(df_propre[['name']].drop_duplicates())

                                 name
0               Course à pied de nuit
1                     10*400 moy 3:01
2                     10*400 Moy 3:01
4             Course à pied en soirée
8     Course à pied dans l'après-midi
9          Bla-bla run avec les girls
14             Course à pied le matin
17                         Pas facile
36    Pause au involontaire au milieu
74             Course à pied matinale
85   Course à pied le matin avec lulu
109            Entraînement en soirée
112             Course à pied le midi
187                         Montre hs
258                    Trail le matin
314           Trail dans l'après-midi
336             Course à pied du midi
473                     Trail le midi


Converstion des valeurs : 

In [43]:
print(df_propre[['distance','moving_time']])


     distance  moving_time
0      7547.9         1822
1      9561.0         2283
2     12299.0         2828
3     16425.0         3717
4      8721.5         2130
..        ...          ...
565    4994.0         1613
566       6.0            8
567   10000.0         2274
569    4995.0         1013
570       8.0            5

[484 rows x 2 columns]


In [44]:
# Conversion de la distance (m -> km)
if 'distance' in df_propre.columns:
    df_propre['distance_km'] = df_propre['distance'] / 1000
    # Conversion du temps total (sec -> min)
if 'elapsed_time' in df_propre.columns:
    df_propre['elapsed_time_min'] = df_propre['elapsed_time'] / 60

# Optionnel : Conversion du temps de mouvement (sec -> min)
if 'moving_time' in df_propre.columns:
    df_propre['moving_time_min'] = df_propre['moving_time'] / 60

In [45]:
print(df_propre[['distance_km','moving_time_min']])


     distance_km  moving_time_min
0         7.5479        30.366667
1         9.5610        38.050000
2        12.2990        47.133333
3        16.4250        61.950000
4         8.7215        35.500000
..           ...              ...
565       4.9940        26.883333
566       0.0060         0.133333
567      10.0000        37.900000
569       4.9950        16.883333
570       0.0080         0.083333

[484 rows x 2 columns]


In [ ]:
# En prévsion d'une  analyse temporelle, on convertit la date de début en format datetime

In [50]:
# Convertir la colonne texte en vrai format Datetime (gère automatiquement les T et Z)
df_propre['start_date_local'] = pd.to_datetime(df_propre['start_date_local'])

# Vérification du nouveau type
print(df_propre['start_date_local'].dtype)
# 1. Extraire l'année (ex: 2026, 2024)
df_propre['annee'] = df_propre['start_date_local'].dt.year

# 2. Extraire le mois sous forme de chiffre (1 à 12)
df_propre['mois_num'] = df_propre['start_date_local'].dt.month

# 3. Extraire le nom du jour en français (lundi, mardi...)
df_propre['jour_semaine'] = df_propre['start_date_local'].dt.day_name()

# 4. Extraire uniquement la date brute (sans les heures/minutes) pour l'affichage
df_propre['date_seule'] = df_propre['start_date_local'].dt.date

# On jette un œil aux nouvelles colonnes créées
df_propre[['start_date_local', 'annee', 'mois_num', 'jour_semaine']].head()

datetime64[us, UTC]


,start_date_local,annee,mois_num,jour_semaine
0,2026-05-15 21:10:05+00:00,2026,5,Friday
1,2026-05-15 19:53:54+00:00,2026,5,Friday
2,2026-05-14 21:49:15+00:00,2026,5,Thursday
3,2026-05-13 22:00:24+00:00,2026,5,Wednesday
4,2026-05-12 20:16:58+00:00,2026,5,Tuesday


In [48]:
print(df_propre['start_date_local'])

0      2026-05-15T21:10:05Z
1      2026-05-15T19:53:54Z
2      2026-05-14T21:49:15Z
3      2026-05-13T22:00:24Z
4      2026-05-12T20:16:58Z
               ...         
565    2024-10-30T21:35:16Z
566    2024-10-30T21:34:32Z
567    2024-10-30T20:44:59Z
569    2024-10-29T17:50:30Z
570    2024-10-29T17:48:37Z
Name: start_date_local, Length: 484, dtype: str


In [47]:
print(df['average_heartrate'].head())

0    165.8
1    162.3
2    163.9
3    164.8
4    152.8
Name: average_heartrate, dtype: float64


In [ ]:
# On enrichie le  dataFrame  par l'ajout de nouvelles colonnes de calcule de VMA, VO2max, indice d'éffort. 